In [1]:
import random
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

random.seed(42)
np.random.seed(42)

CATEGORIES = ["grocery", "food_delivery", "recharge", "bill_payment", "travel",
              "ecommerce", "entertainment"]
REGIONS = ["North", "South", "East", "West"]
METHODS = ["UPI", "Wallet", "Card", "Netbanking"]
METHOD_WEIGHTS = [0.55, 0.20, 0.15, 0.10]
AMOUNTS_INR = [49, 99, 149, 299, 499, 799, 1499, 2999, 4999]
AMOUNT_WEIGHTS = [0.18, 0.16, 0.14, 0.14, 0.12, 0.10, 0.08, 0.05, 0.03]

# --- 40 merchants ---
merchants = pd.DataFrame({
    "merchant_id": range(1, 41),
    "merchant_name": [f"Merchant_{i:03d}" for i in range(1, 41)],
    "category": [random.choice(CATEGORIES) for _ in range(40)],
    "region": [random.choice(REGIONS) for _ in range(40)],
})

# --- 350 established users, signed up 30-730 days before the window start ---
window_start = datetime(2026, 1, 1)
users = pd.DataFrame({
    "user_id": range(1, 351),
    "signup_date": [window_start - timedelta(days=random.randint(30, 730)) for _ in range(350)],
})

# --- 500 baseline transactions over a 30-day window ---
rows = []
for i in range(500):
    txn_time = window_start + timedelta(
        days=random.randint(0, 29), hours=random.randint(0, 23), minutes=random.randint(0, 59))
    status = np.random.choice(["captured", "failed", "chargeback"], p=[0.92, 0.06, 0.02])
    rows.append({
        "transaction_id": f"TXN{100000+i}",
        "user_id": random.randint(1, 350),
        "merchant_id": random.randint(1, 40),
        "transaction_time": txn_time,
        "amount_inr": np.random.choice(AMOUNTS_INR, p=AMOUNT_WEIGHTS),
        "payment_method": np.random.choice(METHODS, p=METHOD_WEIGHTS),
        "status": status,
        "risk_score": random.randint(0, 100),
    })

# --- inject 15 "burner account" chargeback frauds: brand-new users (< 30 days old) ---
next_user_id = 351
for i in range(15):
    txn_time = window_start + timedelta(days=random.randint(10, 29), hours=random.randint(0, 23))
    signup = txn_time - timedelta(days=random.randint(1, 25))
    users = pd.concat([users, pd.DataFrame([{"user_id": next_user_id, "signup_date": signup}])],
                       ignore_index=True)
    rows.append({
        "transaction_id": f"TXN{200000+i}",
        "user_id": next_user_id,
        "merchant_id": random.randint(1, 40),
        "transaction_time": txn_time,
        "amount_inr": random.choice([999, 1999, 2999, 4999]),
        "payment_method": "Card",
        "status": "chargeback",
        "risk_score": random.randint(70, 100),
    })
    next_user_id += 1

# --- inject 8 velocity-attack clusters: 4 rapid-fire txns each within a 5-minute window ---
for cluster in range(8):
    victim_user = random.randint(1, 350)
    base_time = window_start + timedelta(days=random.randint(0, 29), hours=random.randint(0, 23))
    for k in range(4):
        rows.append({
            "transaction_id": f"TXN{300000 + cluster*4 + k}",
            "user_id": victim_user,
            "merchant_id": random.randint(1, 40),
            "transaction_time": base_time + timedelta(minutes=k),
            "amount_inr": random.choice([299, 399, 499]),
            "payment_method": "Card",
            "status": "captured" if k == 3 else "failed",
            "risk_score": random.randint(60, 95),
        })

ledger = pd.DataFrame(rows)  # 500 + 15 + 32 = 547 rows
merchants.to_csv("merchants.csv", index=False)
users.to_csv("users.csv", index=False)
ledger.to_csv("ledger.csv", index=False)

# --- build the deliberately-discrepant "gateway export" copy for reconciliation ---
gateway = ledger.copy()
n = len(gateway)
missing_idx = np.random.choice(n, size=int(0.05 * n), replace=False)
gateway = gateway.drop(index=missing_idx).reset_index(drop=True)

mismatch_idx = np.random.choice(len(gateway), size=int(0.03 * n), replace=False)
gateway.loc[mismatch_idx, "amount_inr"] = gateway.loc[mismatch_idx, "amount_inr"] + \
    np.random.choice([-100, -50, 50, 100], size=len(mismatch_idx))

extra_rows = []
for i in range(int(0.02 * n)):
    extra_rows.append({
        "transaction_id": f"TXNX{9000+i}", "user_id": random.randint(1, 350),
        "merchant_id": random.randint(1, 40),
        "transaction_time": window_start + timedelta(days=random.randint(0, 29)),
        "amount_inr": random.choice(AMOUNTS_INR), "payment_method": random.choice(METHODS),
        "status": "captured", "risk_score": random.randint(0, 100),
    })
gateway = pd.concat([gateway, pd.DataFrame(extra_rows)], ignore_index=True)

status_idx = np.random.choice(len(gateway), size=int(0.02 * n), replace=False)
gateway.loc[status_idx, "status"] = "failed"

gateway.to_csv("gateway_export.csv", index=False)


In [2]:
import pandas as pd


def reconcile_payments(ledger_df, gateway_df):
    """
    Compare ledger and gateway data.

    Returns:
        missing_in_gateway
        missing_in_ledger
        amount_mismatches
        status_mismatches
    """

    # Transactions in ledger but missing in gateway
    missing_in_gateway = ledger_df[
        ~ledger_df["transaction_id"].isin(gateway_df["transaction_id"])
    ]

    # Transactions in gateway but missing in ledger
    missing_in_ledger = gateway_df[
        ~gateway_df["transaction_id"].isin(ledger_df["transaction_id"])
    ]

    # Merge common transactions
    merged = pd.merge(
        ledger_df,
        gateway_df,
        on="transaction_id",
        suffixes=("_ledger", "_gateway")
    )

    # Amount mismatches
    amount_mismatches = merged[
        merged["amount_inr_ledger"] != merged["amount_inr_gateway"]
    ].copy()

    amount_mismatches["difference"] = (
        amount_mismatches["amount_inr_gateway"]
        - amount_mismatches["amount_inr_ledger"]
    )

    # Status mismatches
    status_mismatches = merged[
        merged["status_ledger"] != merged["status_gateway"]
    ].copy()

    return (
        missing_in_gateway,
        missing_in_ledger,
        amount_mismatches,
        status_mismatches,
    )


def main():
    ledger = pd.read_csv("ledger.csv")
    gateway = pd.read_csv("gateway_export.csv")

    (
        missing_gateway,
        missing_ledger,
        amount_diff,
        status_diff,
    ) = reconcile_payments(ledger, gateway)

    print("========== PAYMENT RECONCILIATION ==========")
    print(f"Missing in Gateway : {len(missing_gateway)}")
    print(f"Missing in Ledger  : {len(missing_ledger)}")
    print(f"Amount Mismatches  : {len(amount_diff)}")
    print(f"Status Mismatches  : {len(status_diff)}")

    # Save outputs
    missing_gateway.to_csv("missing_in_gateway.csv", index=False)
    missing_ledger.to_csv("missing_in_ledger.csv", index=False)
    amount_diff.to_csv("amount_mismatches.csv", index=False)
    status_diff.to_csv("status_mismatches.csv", index=False)

    print("\nCSV reports generated successfully.")


if __name__ == "__main__":
    main()

========== PAYMENT RECONCILIATION ==========
Missing in Gateway : 27
Missing in Ledger  : 10
Amount Mismatches  : 16
Status Mismatches  : 9

CSV reports generated successfully.
